# Analyze Time Scales

Only time-warp constant fit to training+val sets, best config ran on test set. 

Notebook is for interactive plotting and summary

In [ ]:
import numpy as np
import pandas as pd
import yaml
import time
import sys
import os
import os.path as osp
import joblib
import pickle
from itertools import product
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.stattools import acf
# Local modules
from src.utils import read_yml, Dict, time_range, time_intp, plot_styles, plt_acf, plt_pacf, str2time
from src.models import moisture_rnn as mrnn

In [ ]:
# document-safe plotting defaults
FIGSIZE = (6.5,4.5)
DPI = 300
LABEL_SIZE = 14
TICK_SIZE = 12
CBAR_LABEL_SIZE = 13

In [ ]:
# Utilty func to read text file summaries
def read_rep_report(path):
    path = Path(path)
    metrics, meta = {}, {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: 
                continue
            if line.startswith("Median replication index"):
                meta["median_replication_index"] = int(line.split(":")[1])
            elif line.startswith("Seed directory"):
                meta["seed_directory"] = line.split(":",1)[1].strip()
            elif line.startswith("Twarp Params"):
                meta["twarp_params"] = line.split(":",1)[1].strip()
            elif ":" in line:
                k,v = line.split(":",1)
                try: metrics[k.strip()] = float(v)
                except ValueError: pass
    return {**meta, **metrics}

## Data

In [ ]:
conf = Dict(read_yml("etc/thesis_config.yaml"))

In [ ]:
wtest = pd.read_csv("outputs/transfer0/weather_test.csv")

In [ ]:
reps_dir = "outputs/transfer0_reps"
zero_dir = "outputs/zeroshot_10h_reps"

wtrain = pd.read_csv(osp.join("outputs/transfer0/weather_train.csv"))
wtest  = pd.read_csv(osp.join("outputs/transfer0/weather_test.csv"))

In [ ]:
fm1_test    = pd.read_excel(osp.join("data", "processed_data/ok_1h.xlsx"))
fm1_test    = fm1_test[fm1_test.utc_rounded.isin(wtest.utc)]
# fm10_test   = pd.read_excel(osp.join("data", "processed_data/ok_10h.xlsx"))
# fm10_test   = fm10_test[fm10_test.utc_rounded.isin(wtest.utc)]
fm100_test  = pd.read_excel(osp.join("data", "processed_data/ok_100h.xlsx"))
fm100_test  = fm100_test[fm100_test.utc_rounded.isin(wtest.utc)]
fm1000_test = pd.read_excel(osp.join("data", "processed_data/ok_1000h.xlsx"))
fm1000_test = fm1000_test[fm1000_test.utc_rounded.isin(wtest.utc)]

In [ ]:
# Summaries with Median Accuracy across Reps
fm1_med_meta    = read_rep_report(osp.join(reps_dir, "fm1_median_rep_report.txt"))
# fm10_med_meta  = read_rep_report(osp.join(zero_dir, "median_rep_report.txt"))
fm100_med_meta  = read_rep_report(osp.join(reps_dir, "fm100_median_rep_report.txt"))
fm1000_med_meta = read_rep_report(osp.join(reps_dir, "fm1000_median_rep_report.txt"))

In [ ]:
# All Model Reps
results = [
    pd.read_pickle(p / "results_test_set.pkl")
    for p in sorted(
        (p for p in Path(reps_dir).iterdir() if p.is_dir() and p.name.startswith("seed_")),
        key=lambda p: int(p.name.split("_")[1])
    )
]

In [ ]:
# results0 = [
#     pd.read_pickle(p / f"results_zeroshot_{int(p.name.split('_')[1])}.pkl")
#     for p in sorted(
#         (p for p in Path(zero_dir).iterdir() if p.is_dir() and p.name.startswith("seed_")),
#         key=lambda p: int(p.name.split("_")[1])
#     )
# ]

## Models

In [ ]:
# Models with reps
# Subset fm10 to same test set
fm1    = np.stack([r["FM1"]["preds1"] for r in results], axis=0)
# inds10 = np.where((results0[0]["times"] >= str2time(conf.f_start)) & (results0[0]["times"] <= str2time(conf.f_end)))[0]
# fm10   = np.stack([r0["preds"][inds10] for r0 in results0])
fm100  = np.stack([r["FM100"]["preds100"] for r in results])
fm1000 = np.stack([r["FM1000"]["preds1000"] for r in results])

In [ ]:
fm1.shape == fm100.shape == fm1000.shape

## Plot Time Series

In [ ]:
plt_start = pd.Timestamp('1997-08-13 12:00:00')+ pd.DateOffset(hours=12)
plt_end   = plt_start + pd.DateOffset(hours=168-1)
wtest["utc"] = pd.to_datetime(wtest.utc)
inds = np.where((wtest.utc>=plt_start) & (wtest.utc<=plt_end))
x=wtest.utc.iloc[inds]

y = fm1[0,inds].flatten()
fm = fm1_test[(fm1_test.utc_rounded >= plt_start) & (fm1_test.utc_rounded <= plt_end )]

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(x, y, **plot_styles["model1"])
ax.scatter(fm.utc_rounded, fm.fm1, **plot_styles["fm1"])
ax.tick_params(rotation=45)

In [ ]:
plt_start = pd.Timestamp('1997-08-13 12:00:00')+ pd.DateOffset(hours=12)
plt_end   = plt_start + pd.DateOffset(hours=168-1)
wtest["utc"] = pd.to_datetime(wtest.utc)
inds = np.where((wtest.utc>=plt_start) & (wtest.utc<=plt_end))
x=wtest.utc.iloc[inds]

y = fm100[0,inds].flatten()
fm = fm100_test[(fm100_test.utc_rounded >= plt_start) & (fm100_test.utc_rounded <= plt_end )]

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(x, y, **plot_styles["model100"])
ax.scatter(fm.utc_rounded, fm.fm100, **plot_styles["fm100"])
ax.tick_params(rotation=45, labelsize=LABEL_SIZE)

- Start with pretrained FM10 network trained on sensor data and HRRR weather
- Fit time-warp parameters to training set FM1, FM100, FM1000
- Examining predictions in test set to evaluate whether timescale really warped

In [ ]:
plt_start = pd.Timestamp('1997-08-13 12:00:00')+ pd.DateOffset(hours=6)
plt_end   = plt_start + pd.DateOffset(hours=72-1)
wtest["utc"] = pd.to_datetime(wtest.utc)
inds = np.where((wtest.utc>=plt_start) & (wtest.utc<=plt_end))
x=wtest.utc.iloc[inds]

y1 = fm1[1,inds].flatten()
y100 = fm100[1,inds].flatten()
y1000 = fm1000[1,inds].flatten()

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(x, y1, **plot_styles["model1"], linewidth=2.5, marker="o", markevery=5)
# ax.plot(x, y10, **plot_styles["model"], linewidth=2.5, marker="s", markevery=5)
ax.plot(x, y100, **plot_styles["model100"], linewidth=2.5, marker='^', markevery=5)
ax.plot(x, y1000, **plot_styles["model1000"], linewidth=2.5, marker="D", markevery=5)
ax.legend()
ax.tick_params(rotation=45, labelsize=LABEL_SIZE)
plt.tight_layout()
plt.savefig("outputs/tscales.png")

## Compare Derivatives

In [ ]:
t = np.arange(0, fm1.shape[1])

grad_fm1    = np.gradient(fm1, t, axis=1)
std_grad_fm1 = np.std(grad_fm1, axis=1)

# grad_fm10   = np.gradient(fm10, t, axis=1)
# std_grad_fm10 = np.std(grad_fm10, axis=1)

grad_fm100  = np.gradient(fm100, t, axis=1)
std_grad_fm100 = np.std(grad_fm100, axis=1)

grad_fm1000 = np.gradient(fm1000, t, axis=1)
std_grad_fm1000 = np.std(grad_fm1000, axis=1)

In [ ]:
models = {
    "FM1": grad_fm1,
    # "FM10": grad_fm10,
    "FM100": grad_fm100,
    "FM1000": grad_fm1000
}

rows = []

for name, g in models.items():
    mean_grad = np.mean(g, axis=1)
    std_grad  = np.std(g, axis=1)

    mean_mean = np.mean(mean_grad)
    std_mean  = np.std(mean_grad)

    mean_std  = np.mean(std_grad)
    std_std   = np.std(std_grad)

    rows.append({
        "Model": name,
        "Mean Gradient Mean": f"{mean_mean:.5f}",
        "Mean Gradient Std":  f"{std_mean:.5f}",
        "Std Gradient Mean":  f"{mean_std:.5f}",
        "Std Gradient Std":   f"{std_std:.5f}"
    })

gtable = pd.DataFrame(rows)
gtable

## ACF 

In [ ]:
acf_list = [acf(fm1[i], nlags=200) for i in range(fm1.shape[0])]
acf1 = np.vstack(acf_list)

acf_list = [acf(fm100[i], nlags=200) for i in range(fm100.shape[0])]
acf100 = np.vstack(acf_list)

acf_list = [acf(fm1000[i], nlags=200) for i in range(fm1000.shape[0])]
acf1000 = np.vstack(acf_list)

In [ ]:
# extract indices for median models
ind1    = fm1_med_meta["median_replication_index"]
ind100  = fm100_med_meta["median_replication_index"]
ind1000 = fm1000_med_meta["median_replication_index"]

In [ ]:
plt_acf(acf1[ind1,:], title="FM1 Autocorrelation",
        color=plot_styles["fm1"]["color"], save_path="outputs/fm1_acf.png")

In [ ]:
plt_acf(acf100[ind100,:], title="FM100 Autocorrelation",
        color=plot_styles["fm100"]["color"], save_path="outputs/fm100_acf.png")

In [ ]:
plt_pacf(acf1[ind1,:], max_k=30, title="FM1 Partial-Autocorrelation",
         color=plot_styles["fm1"]["color"], save_path="outputs/fm1_pacf.png")

In [ ]:
plt_pacf(acf100[ind100,:], max_k=30, title="FM100 Partial-Autocorrelation",
         color=plot_styles["fm100"]["color"], save_path="outputs/fm100_pacf.png")

In [ ]:
plt_acf(acf1000[ind1000,:], title="FM1000 Autocorrelation",
        color=plot_styles["fm1000"]["color"], save_path="outputs/fm1000_acf.png")

In [ ]:
plt_pacf(acf1000[ind1000,:], max_k=30, title="FM1000 Partial-Autocorrelation",
         color=plot_styles["fm1000"]["color"], save_path="outputs/fm1000_pacf.png")